# ERA5 to PRISM Downscaling (Georgia)
Multi-variable regional downscaling with persistence, linear, CNN, and ConvLSTM.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


## Problem and setup
- ERA5 is coarse reanalysis input.
- PRISM is higher-resolution temperature target.
- Input channels: `t2m`, `u10`, `v10`, `sp`.
- Mapping: `ERA5(t-k+1 ... t) -> PRISM(t)`.


## Models
Persistence and linear baselines are references. CNN is the spatial learned model. ConvLSTM is the temporal model.

In [ ]:
import json
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from datasets.prism_dataset import ERA5_PRISM_Dataset

ERA5_PATH = ROOT / "data_raw/era5_georgia_temp.nc"
PRISM_DIR = ROOT / "data_raw/prism"
RESULTS_DIR = ROOT / "results/evaluation"

if ERA5_PATH.exists() and PRISM_DIR.exists():
    era5_ds = xr.open_dataset(ERA5_PATH)
    print("ERA5 variables:", list(era5_ds.data_vars))
    ds = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=6, verbose=False)
    x, y = ds[0]
    print("Usable samples (history=6):", len(ds))
    print("Input shape [T, C, H, W]:", tuple(x.shape))
    print("Target shape [C, H, W]:", tuple(y.shape))
else:
    print("Data is missing. Run:")
    print("python data_pipeline/download_era5_georgia.py --year 2023 --month 1")
    print("python data_pipeline/download_prism.py --start-date 20230101 --days 30 --variable tmean")


## Results (history=6)

In [ ]:
def latest_png(folder: Path):
    files = sorted(folder.glob("comparison_*.png"))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

cnn_img = latest_png(RESULTS_DIR / "cnn")
convlstm_img = latest_png(RESULTS_DIR / "convlstm")
summary_csv = RESULTS_DIR / "baselines_summary.csv"
model_comparison = RESULTS_DIR / "model_comparison.png"

if cnn_img:
    display(Markdown("### CNN"))
    display(Image(filename=str(cnn_img)))
if convlstm_img:
    display(Markdown("### ConvLSTM"))
    display(Image(filename=str(convlstm_img)))
if model_comparison.exists():
    display(Markdown("### Model comparison"))
    display(Image(filename=str(model_comparison)))

if summary_csv.exists():
    display(Markdown("### Metrics table"))
    display(pd.read_csv(summary_csv))
else:
    print("Run:")
    print("python training/train_downscaler.py --model cnn --history-length 6 --epochs 30 --learning-rate 1e-3 --split-seed 42 --grad-clip 1.0")
    print("python training/train_downscaler.py --model convlstm --history-length 6 --epochs 40 --learning-rate 3e-4 --split-seed 42 --grad-clip 1.0")
    print("python evaluation/evaluate_model.py --models persistence linear cnn convlstm --history-length 6 --num-samples 8 --split-seed 42")


## Effect of Temporal Depth

In [ ]:
history3 = pd.DataFrame([
    {"model": "persistence", "rmse": 3.2510, "mae": 2.6106},
    {"model": "linear", "rmse": 2.9649, "mae": 2.6049},
    {"model": "cnn", "rmse": 3.9735, "mae": 3.4459},
    {"model": "convlstm", "rmse": 3.0981, "mae": 2.5130},
])
history3["history"] = 3

if (RESULTS_DIR / "baselines_summary.csv").exists():
    history6 = pd.read_csv(RESULTS_DIR / "baselines_summary.csv")[["model", "rmse", "mae"]].copy()
    history6["history"] = 6
    compare = pd.concat([history3, history6], ignore_index=True)
    display(compare.pivot(index="model", columns="history", values=["rmse", "mae"]).round(4))
else:
    print("Run evaluation first to build history=6 table.")


- Increasing temporal window from 3 to 6 gives ConvLSTM more sequence context.
- In the current run, ConvLSTM improved enough to become competitive with and slightly better than linear on RMSE/MAE.
- Longer windows help, but results are still tied to data size and date coverage.
